# Bomber Arena Student Colab
Q-learning の学習ループ、報酬設計、W&B/CSV ログ、提出物生成までを1本で確認します。

In [ ]:
# Colabでは必要に応じて有効化
# !git clone <course-repo-url>
# %cd <course-repo>
# !pip install -e ".[train]"
# !wandb login


In [ ]:
from pathlib import Path
import math, json
from marl_course.algos.bomber_qlearning import BomberQLearningPolicy
from marl_course.algos.logging import MetricLogger
from marl_course.envs.bomber_arena import BomberArenaEnv, BomberRuleBasedPolicy


In [ ]:
config = json.loads(Path('configs/train_bomber.json').read_text())
config.update({
    'episodes': 50,
    'out': 'outputs/colab_bomber',
    'wandb': False  # W&Bを使う場合は True
})
out = Path(config['out'])


In [ ]:
# 学生が自由に変更する学習用報酬。教師評価では使われません。
def student_reward(env, events):
    reward = 0.0
    for e in events:
        if e.name == 'winner' and e.actor == 'agent_0':
            reward += 1.0
        elif e.name == 'powerup_collected' and e.actor == 'agent_0':
            reward += 0.05
        elif e.name == 'block_destroyed':
            reward += 0.01
        elif e.name == 'self_eliminated' and e.actor == 'agent_0':
            reward -= 0.5
    if env.last_winner is not None and env.last_winner != 'agent_0':
        reward -= 0.2
    return reward


In [ ]:
policy = BomberQLearningPolicy(seed=config['seed'])
logger = MetricLogger(out, use_wandb=config['wandb'], project=config['wandb_project'], run_name='colab-bomber-qlearning', config=config)
wins = 0
moving_win = 0.0
for ep in range(config['episodes']):
    epsilon = config['epsilon_end'] + (config['epsilon_start'] - config['epsilon_end']) * math.exp(-3.0 * ep / max(1, config['episodes']))
    env = BomberArenaEnv()
    opponents = [BomberRuleBasedPolicy(seed=config['seed'] + ep * 10 + i) for i in range(3)]
    obs, _ = env.reset(seed=config['seed'] + ep)
    done = False
    total_reward = 0.0
    td_error = 0.0
    updates = 0
    while not done:
        agent_obs = obs['agent_0']
        action = policy.act(agent_obs, agent_obs['action_mask'], deterministic=False, epsilon=epsilon)
        actions = {'agent_0': action}
        for idx in range(1, 4):
            actions[f'agent_{idx}'] = opponents[idx - 1].act(obs[f'agent_{idx}'], obs[f'agent_{idx}']['action_mask'])
        result = env.step(actions)
        done = any(result.terminations.values()) or any(result.truncations.values())
        reward = student_reward(env, result.infos['agent_0']['events'])
        total_reward += reward
        td_error += policy.update(agent_obs, action, reward, result.observations['agent_0'], done, config['alpha'], config['gamma'])
        updates += 1
        obs = result.observations
    win = int(env.last_winner == 'agent_0')
    wins += win
    moving_win = 0.95 * moving_win + 0.05 * win
    logger.log({'episode': ep, 'return': total_reward, 'win': win, 'moving_win_rate': moving_win, 'epsilon': epsilon, 'steps': env.step_count, 'td_error': td_error / max(1, updates), 'q_states': len(policy.q_table)}, step=ep)
logger.close()
print('win_rate', wins / config['episodes'])


In [ ]:
out.mkdir(parents=True, exist_ok=True)
policy.save(out / 'policy.pt')
(out / 'metadata.json').write_text(json.dumps({'student_id': 'colab_bomber', 'env_id': 'bomber_arena_v1'}, indent=2), encoding='utf-8')
(out / 'policy.py').write_text("from pathlib import Path\nfrom marl_course.algos.bomber_qlearning import BomberQLearningPolicy\n\ndef load_policy(model_path, device='cpu'):\n    return BomberQLearningPolicy.load(Path(model_path))\n", encoding='utf-8')
print('submission:', out)


In [ ]:
import pandas as pd
pd.read_csv(out / 'metrics.csv').plot(x='episode', y=['return', 'moving_win_rate', 'epsilon'], subplots=True, figsize=(8, 6));
